#Integrantes: Javier Nicolás y Jhonatan Arturo Elnathan Carreño Prieto

**Situación 1 — Outlier en `dias_ausencia_anio`**

Hay un trabajador con 187 días de ausencia en el año. Un año laboral en Colombia tiene aproximadamente 240 días hábiles.

Tarea: detecten ese outlier con IQR, decidan qué hacer con él y justifiquen su decisión en un comentario en el notebook.

```python
# Detecten el outlier con IQR
# Decidan: ¿eliminar la fila, corregir el valor, o conservarlo?
Se elige corregir el valor 
# Escriban la justificación como comentario en esta celda
Ya que el calculo arrojo que el limite inferior es negativo lo cual en horas laborales no tiene sentido que un trabajador tenga tantos dias de ausencia al año sin ningun aviso por lo que creemos que haya sido un error.
```

**Situación 2 — Decisión de imputación para `nivel_estres`**

Volvamos a `nivel_estres`. Hay 28 nulos. Antes de imputar, comparen: ¿el nivel de estrés promedio difiere entre Juniors y Seniors?

Tarea: calculen la media de estrés por nivel, decidan si imputar con la media global o por grupo, y ejecuten la imputación.

```python
# Calculen media de nivel_estres por nivel
# Decidan la estrategia de imputación
Se decidio la media global porque se mantiene mas la coherencia en analisis de datos y al hacerlo por grupo no cambia mucho
# Ejecuten y verifiquen que no queden nulos
```

**Situación 3 — Reflexión sin código**

¿Qué columna del dataset les preocupa más en términos de calidad de datos y por qué? Escriban una respuesta de 2–3 líneas en una celda Markdown del notebook.

In [1]:
import pandas as pd
import numpy as np

print(pd.__version__)
print(np.__version__)

3.0.1
2.4.4


In [2]:
df = pd.read_csv("healthcheck_colombia.csv")
print(df.shape)
df.head(10)

(350, 13)


,id_trabajador,edad,ciudad,cargo,sector,nivel,anios_experiencia,horas_semana,salario_cop,nivel_estres,dias_ausencia_anio,satisfaccion_laboral,fecha_ingreso
0,1001,28,Bogotá,QA Engineer,Fintech,Junior,1,48.0,10859922.0,2.0,10.0,6.0,01/01/2018
1,1002,41,Barranquilla,UX Designer,Consultoría,Junior,9,56.0,6190583.0,8.0,6.0,8.0,04/01/2018
2,1003,50,Barranquilla,Desarrollador Junior,E-commerce,Lead,11,50.0,4099908.0,6.0,4.0,6.0,07/01/2018
3,1004,36,Bogotá,UX Designer,Healthtech,Senior,16,46.0,4664412.0,4.0,16.0,9.0,10/01/2018
4,1005,32,Medellín,Desarrollador Junior,Healthtech,Mid,7,53.0,7809792.0,5.0,19.0,9.0,13/01/2018
5,1006,29,Bogotá,Data Analyst,Edtech,Mid,10,58.0,4853847.0,8.0,8.0,2.0,16/01/2018
6,1007,50,Barranquilla,QA Engineer,Edtech,Mid,2,43.0,5392478.0,3.0,13.0,10.0,19/01/2018
7,1008,42,Bucaramanga,DevOps,Consultoría,Junior,5,45.0,NaN,NaN,11.0,1.0,22/01/2018
8,1009,28,Bucaramanga,Product Manager,Consultoría,Lead,8,49.0,8449852.0,4.0,5.0,2.0,25/01/2018
9,1010,47,Cali,Desarrollador Junior,Fintech,Mid,5,46.0,2529725.0,1.0,16.0,5.0,28/01/2018


In [8]:
Q1 = df ['dias_ausencia_anio'].quantile(0.25)
Q3 = df ['dias_ausencia_anio'].quantile(0.75)

IQR = Q3 - Q1

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR}")

# Limite Inferior y Superior

L_I = Q1 - (1.5 * IQR)
print(f"El limite Inferior es: {L_I:.2f} horas")

L_S = Q3 + (1.5 * IQR)
print(f"El limite Superior es: {L_S:.2f} horas")

outliers_iqr = df[
(df['dias_ausencia_anio'] < L_I) |
(df['dias_ausencia_anio'] > L_S)
]
print(f"Outliers detectados en horas_semana: {len(outliers_iqr)}")
print(outliers_iqr[['id_trabajador', 'cargo', 'nivel', 'dias_ausencia_anio']])


Q1: 5.0
Q3: 14.0
IQR: 9.0
El limite Inferior es: -8.50 horas
El limite Superior es: 27.50 horas
Outliers detectados en horas_semana: 1
     id_trabajador            cargo nivel  dias_ausencia_anio
289           1290  Product Manager   Mid               187.0


In [10]:
media_por_nivel = df.groupby('nivel')['nivel_estres'].mean()
print(media_por_nivel)

media_global = df['nivel_estres'].mean()
print(media_global)

diferencia = abs(media_por_nivel.max() - media_por_nivel.min())
print(diferencia)

if diferencia > 0.5:
    df['nivel_estres'] = df.groupby('nivel')['nivel_estres'].transform(
    lambda x: x.fillna(x.mean())
    )
else:
    df['nivel_estres'] = df['nivel_estres'].fillna(media_global)

print(df['nivel_estres'].isnull().sum())

nivel
Junior    5.099099
Lead      5.485714
Mid       5.273585
Senior    5.385714
Name: nivel_estres, dtype: float64
5.260869565217392
0.38661518661518635
0


¿Qué columna del dataset les preocupa más en términos de calidad de datos y por qué? Escriban una respuesta de 2–3 líneas en una celda Markdown del notebook.

Ejecutando el comando "df.types" se encuentra que la celda fecha_ingreso es de tipo "string" lo cual en el uso de un dataset que no se pueda ejecutar con formulas debido al tipo de dato que es hace mas complejo a veces cambiar o calcular valores en el resto de la tabla, ademas la columna de niveles de estrés  puede llegar a ser una postura más compleja de trabajar ya que se convierte en un análisis más sugestivo donde el estado de ánimo y la influencia externa termina generando una variación en los datos lo que genera pequeñas variaciones y distorsiona los resultados.